In [32]:
import numpy as np
from CifFile import ReadCif
from diffpy.structure import load_structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.crystal_map import Phase
from orix.quaternion import Rotation
from diffsims.generators.zap_map_generator import get_rotation_from_z_to_direction

In [2]:
import pint

## ICSD data

Gold (111) is the largest reflection with finite intensity

d-spacing (111) is 2.3503 Angstrom

At wavelength of 0.3 Angstrom: Twotheta equals 7.3 deg

## https://www.globalsino.com/EM/page4171.html

Table 4171:

hkl 111 0.2350 nm 200 kV twotheta 10.68 mrad

## Bragg condition

https://www.jeol.com/words/emterms/20121023.071258.php#gsc.tab=0

Wavelength at 200 kV: 2.5079 pm

n * lambda = 2 * d sin (theta)

sin x ca x (small diffraction angle):

2theta = n * lambda / d


In [10]:
(1 * pint.Quantity(2.5079, 'pm').to('m') / pint.Quantity(0.2350, 'nm').to('m')).magnitude * 1000

10.67191489361702

--> Bragg condition calculation based on Jeol wavelength and ICSD + globalsino d-spacing matches globalsino value

In [17]:
structure_raw = ReadCif('AuEntryWithCollCode163723.cif')
key = list(structure_raw.keys())[0]
space_group = int(structure_raw[key]["_space_group_IT_number"])
structure = load_structure('AuEntryWithCollCode163723.cif')
p = Phase(structure=structure, space_group=space_group)

In [112]:
structure.lattice

Lattice(a=4.0709, b=4.0709, c=4.0709, alpha=90, beta=90, gamma=90)

In [115]:
structure.lattice.a / np.sqrt(3)

np.float64(2.350335210844048)

In [43]:
zone_axis = [1, 1, 1]
eulers = get_rotation_from_z_to_direction(p.structure, zone_axis)

In [44]:
eulers

array([ 135.        ,   54.73561032, -135.        ])

In [85]:
gen = SimulationGenerator(
    accelerating_voltage=200,
    precession_angle=10,
    minimum_intensity=1,    
)

In [86]:
sim = gen.calculate_diffraction2d(
    phase=p,
    reciprocal_radius=1,
    # rotation=Rotation.from_euler(eulers),
    # Large excitation error to capture many peaks
    max_excitation_error=1e9,
)

In [87]:
sim.coordinates.calculate_theta(voltage=200000)

In [88]:
[item for item in zip(sim.coordinates.hkl.astype(int), sim.coordinates.theta, sim.coordinates.dspacing) if np.all(item[0] == 1)]

[]

In [106]:
gen = SimulationGenerator(
    accelerating_voltage=200,
    precession_angle=10,
    minimum_intensity=0.0001,
)

for h in (-2, -1, 0, 1, 2):
    for k in (-2, -1, 0, 1, 2):
        for l in (-2, -1, 0, 1, 2):
            eulers = get_rotation_from_z_to_direction(p.structure, [h, k, l])
            sim = gen.calculate_diffraction2d(
                phase=p,
                reciprocal_radius=1,
                rotation=Rotation.from_euler(eulers),
                max_excitation_error=0.0001,
            )
            sim.coordinates.calculate_theta(voltage=200000)
            print([item for item in zip(sim.coordinates.hkl.astype(int), sim.coordinates.theta, sim.coordinates.dspacing) if np.all(np.abs(item[0]) == 1)])

[]
[(array([1, 1, 1]), np.float64(0.005334829017673419), np.float64(2.3503352104835065)), (array([-1, -1, -1]), np.float64(0.005334829017673419), np.float64(2.3503352104835065))]
[(array([ 1, -1,  1]), np.float64(0.005334829017673419), np.float64(2.350335210483506)), (array([-1,  1, -1]), np.float64(0.005334829017673419), np.float64(2.350335210483506))]
[(array([ 1, -1, -1]), np.float64(0.005334829017673419), np.float64(2.3503352104835065)), (array([-1,  1,  1]), np.float64(0.005334829017673419), np.float64(2.3503352104835065))]
[(array([ 1, -1, -1]), np.float64(0.005334829017673419), np.float64(2.350335210483506)), (array([-1,  1,  1]), np.float64(0.005334829017673419), np.float64(2.350335210483506))]
[]
[]
[(array([ 1, -1, -1]), np.float64(0.005334829017673419), np.float64(2.350335210483506)), (array([-1,  1,  1]), np.float64(0.005334829017673419), np.float64(2.350335210483506))]
[]
[(array([ 1, -1,  1]), np.float64(0.005334829017673419), np.float64(2.350335210483506)), (array([-1,  

In [109]:
def get_twothetas(cif_filename, acceleration_voltage_V, reciprocal_radius=3):
    gen = SimulationGenerator(
        accelerating_voltage=acceleration_voltage_V / 1000,
        precession_angle=10,
        minimum_intensity=0.0001,
    )
    structure_raw = ReadCif(cif_filename)
    key = list(structure_raw.keys())[0]
    space_group = int(structure_raw[key]["_space_group_IT_number"])
    structure = loadStructure(cif_filename)
    p = Phase(structure=structure, space_group=space_group)
    thetas = set()
    for h in (-1, 0, 1,):
        for k in (-1, 0, 1,):
            for l in (-1, 0, 1):
                euler = get_rotation_from_z_to_direction(p.structure, [h, k, l])
                rot = Rotation.from_euler(euler)
                sim = gen.calculate_diffraction2d(
                    phase=p,
                    rotation=rot,
                    reciprocal_radius=reciprocal_radius,
                    max_excitation_error=0.0001,
                )
                sim.coordinates.calculate_theta(voltage=acceleration_voltage_V)
                thetas_with_intensity = [
                    item[1] for item in zip(sim.coordinates.intensity, sim.coordinates.theta) if item[0] > 1
                ]
                thetas.update(np.round(thetas_with_intensity, decimals=5))
    # diffsims seems to calculate theta, not twotheta
    return np.array(sorted(thetas)) * 2


In [110]:
get_twothetas('AuEntryWithCollCode163723.cif', 200000)

/tmp/ipykernel_2562810/25529549.py:10: DeprecationWarning: 'diffpy.structure.loadStructure' is deprecated and will be removed in version 4.0.0. Please use 'diffpy.structure.load_structure' instead.
  structure = loadStructure(cif_filename)


array([0.     , 0.01066, 0.01232, 0.01742, 0.02044, 0.02686, 0.02754])